In [ ]:
import os
import subprocess
import sys
import traceback


print("=" * 60)
print("NEMOTRON LORA v41 (v20 kernel, v40 code)")
print("Fix: Using v20 kernel which already has model attached")
print("=" * 60)

try:
    import kagglehub
    import pandas as pd
    import torch
    from datasets import Dataset
    from peft import LoraConfig, TaskType, get_peft_model
    from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments

    print("[1/7] Dependencies loaded")

    # Hardware
    print("\n[2/7] Hardware:")
    if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            prop = torch.cuda.get_device_properties(i)
            print(f"  GPU {i}: {prop.name} ({prop.total_memory / 1024**3:.1f} GB)")

    # Load data
    print("\n[3/7] Loading training data...")
    train_file = None
    for base_path in ["/kaggle/input/nvidia-nemotron-model-reasoning-challenge", "/kaggle/input"]:
        if os.path.exists(base_path):
            for root, dirs, files in os.walk(base_path):
                for f in files:
                    if f.lower() == 'train.csv':
                        train_file = os.path.join(root, f)
                        break
                if train_file: break
        if train_file: break

    if not train_file:
        raise FileNotFoundError("train.csv not found")

    df = pd.read_csv(train_file)
    print(f"  Loaded {len(df)} rows")

    # Prepare training texts
    print("\n[4/7] Preparing training data...")
    training_texts = []
    for idx, row in df.iterrows():
        prompt = row['prompt']
        answer = str(row['answer']).strip()
        text = f"Problem: {prompt}\n\nLet's solve this step by step.\n\nTherefore, the answer is \\boxed{{{answer}}}."
        training_texts.append(text)
        if (idx + 1) % 2000 == 0:
            print(f"  {idx+1}/{len(df)}")

    # Load model via kagglehub (model ALREADY ATTACHED to this kernel)
    print("\n[5/7] Loading Nemotron model (pre-attached)...")
    model_path = kagglehub.model_download("metric/nemotron-3-nano-30b-a3b-bf16/transformers/default")
    tokenizer = AutoTokenizer.from_pretrained(model_path, trust_remote_code=True)
    if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        model_path, device_map="auto", trust_remote_code=True, torch_dtype=torch.bfloat16
    )

    lora_config = LoraConfig(
        r=32, lora_alpha=16,
        target_modules=["in_proj", "out_proj", "up_proj", "down_proj"],
        lora_dropout=0.05, bias="none", task_type=TaskType.CAUSAL_LM
    )
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()

    # Tokenize
    print("\n[6/7] Tokenizing...")
    def tokenize_fn(examples):
        return tokenizer(examples["text"], truncation=True, max_length=1024, padding="max_length")

    dataset = Dataset.from_dict({"text": training_texts})
    tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
    tokenized = tokenized.map(lambda x: {"labels": x["input_ids"]}, batched=True)
    print(f"  Dataset: {len(tokenized)} examples")

    # Train
    print("\n[7/7] Training...")
    args = TrainingArguments(
        output_dir="./nemotron_lora_adapter",
        num_train_epochs=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=8,
        learning_rate=1e-4,
        bf16=True,
        gradient_checkpointing=True,
        logging_steps=50,
        save_strategy="no",
        report_to="none",
    )

    trainer = Trainer(model=model, args=args, train_dataset=tokenized)
    trainer.train()

    # Save
    model.save_pretrained("./nemotron_lora_adapter")
    tokenizer.save_pretrained("./nemotron_lora_adapter")
    subprocess.run("cd nemotron_lora_adapter && zip -r ../submission.zip ./*", shell=True, check=True)
    print("\n" + "=" * 60)
    print("SUBMISSION READY: submission.zip")
    print("=" * 60)

except Exception as e:
    print(f"\nERROR: {e}")
    traceback.print_exc()
    sys.exit(1)
